<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter Notebook is part of a series of computer laboratories that are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-512 Image Processing II**](https://moodle.epfl.ch/course/view.php?id=522) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](http://bigwww.epfl.ch/). 
The distribution or reproduction of the notebook is strictly prohibited without the written consent of the authors.  &copy; EPFL 2026.

**Authors**: 
    Sébastien Herbreteau,
    Sepand Kashani,
    [Zhiyuan Hu](mailto:zhiyuan.hu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch)
    
To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 7.2: Introduction to DeepInverse
**Released**: Thursday, May 21, 2026

**Submission deadline**: Monday, June 1, 2026, before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=463)

**Grade weight**: Lab 7 (21 points), 7.5 % of the overall grade

**Related lectures**: Chapter 9 and 10

### Student Name: 

### SCIPER: 

Double-click on this cell and fill your name and SCIPER number. Then, run the cell below to verify your identity in Noto and set the seed for random results.

In [ ]:
import getpass
# This line recovers your camipro number to mark the images with your ID
uid = int(getpass.getuser().split('-')[2]) if len(getpass.getuser().split('-')) > 2 else ord(getpass.getuser()[0])
print(f'SCIPER: {uid}')

## Imports
In the next cell, we import Python libraries that  we will use throughout the lab.

In [ ]:
# Import standard required packages for this exercise
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import skimage
from interactive_kit import imviewer as viewer

import torch
import deepinv as dinv

# Configure plotting as dynamic
%matplotlib widget

# Introduction to DeepInverse (7 points)

## DeepInverse in a Nutshell (0 points)

[DeepInverse](https://deepinv.github.io/deepinv/) is an open source software library to model computational imaging pipelines and solve inverse problems.
It is built on PyTorch and organized around the core concepts of **physics** (forward operators) and **optimizers** (solvers).

Physics operators define a transformation from an input image space to a measurement space.
In the context of computational imaging or signal processing, we use the term **physics** to describe the chain of transformations that go from the object of interest (an image) to the measured data.
DeepInverse comes with a large collection of built-in operators to model various imaging pipelines.
They can be composed using the `ComposedLinearPhysics` class:

```python
>> physics = dinv.physics.ComposedLinearPhysics(A1, A2)  # A2(A1(x))
>> y = physics(x)            # Forward operator
>> x = physics.A_adjoint(y)  # Adjoint operator
>> x = physics.A_dagger(y)   # Pseudo-inverse
```

**Note:** All images in DeepInverse are PyTorch tensors of shape `(B, C, H, W)` where `B` is the batch size, `C` is the number of channels, and `H, W` are the spatial dimensions.

#### How to use a physics operator

Physics operators are instances of `deepinv.physics.Physics` or its subclasses.
Their main interface consists of:
- `A(x)`: forward operator (same as calling `physics(x)`).
- `A_adjoint(y)`: adjoint of the linear operator.
- `A_dagger(y)`: pseudo-inverse (e.g. filtered back-projection for Radon).

For example, consider the situation where we have a measurement $y$ which is the realization of a forward model $A(x)$, where $x$ is to be determined. If $A$ is a masked convolution, we can express it as the composition of a convolution followed by pixelwise masking (ignore the warning about device):

In [ ]:
kernel = skimage.io.imread('images/psf_movement.tif').astype(np.float64)
# Convert kernel to torch tensor: (1, 1, kH, kW)
kernel_t = torch.tensor(kernel, dtype=torch.float32)[None, None]

# Blur operator for (1, 512, 512) images with reflect padding
ConvOp = dinv.physics.Blur(filter=kernel_t, padding='reflect')

np.random.seed(1234) # for reproducibility, do not modify
mask = np.random.binomial(n=1, p=0.95, size=(512, 512))
# Mask tensor: (1, 512, 512) broadcastable to (1, 1, 512, 512)
mask_t = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
# Inpainting operator for (1, 512, 512) images
MaskOp = dinv.physics.Inpainting(img_size=(1, 512, 512), mask=mask_t)

# Compose operators: ConvOp first, then MaskOp (i.e. A(x) = Mask(Conv(x)))
physics = dinv.physics.ComposedLinearPhysics(ConvOp, MaskOp)

In DeepInverse, `ComposedLinearPhysics(A1, A2, ...)` applies the operators sequentially: $A(x) = A_n(\ldots A_2(A_1(x)))$.
The adjoint is automatically computed as $A^\top(y) = A_1^\top(A_2^\top(\ldots A_n^\top(y)))$.

Thus defined, $A$ is a **physics operator** which can be applied to any image of size $512 \times 512$.

**Note:** In DeepInverse, images are PyTorch tensors with batch and channel dimensions `(B, C, H, W)`. We use `[None, None]` to add the batch and channel dimensions to two-dimenstional numpy arrays.

In [ ]:
einstein = skimage.io.imread('images/einstein.tif').astype(np.float64)
# Convert to torch tensor: (1, 1, H, W)
einstein_t = torch.tensor(einstein, dtype=torch.float32)[None, None]

masked_and_blurred_einstein_t = physics(einstein_t)
masked_and_blurred_and_noisy_einstein_t = masked_and_blurred_einstein_t + 10 * torch.randn_like(masked_and_blurred_einstein_t)

# Squeeze back to 2D numpy arrays for display
mbe = masked_and_blurred_einstein_t.squeeze().numpy()
mbne = masked_and_blurred_and_noisy_einstein_t.squeeze().numpy()

plt.close('all')
view = viewer([mbe, mbne, einstein, kernel], widgets=True, hist=False, axis=True, cmap='gray')

#### Data-fidelity terms

Inverse problems are specified by data-fidelity loss functions $f(x; y, A)$.
DeepInverse provides [data-fidelity classes](https://deepinv.github.io/deepinv/api/deepinv.optim.html#data-fidelity) that work together with a physics operator.
A key design choice: in DeepInverse, the data fidelity is **decoupled** from the physics operator and connected at solve time.

In [ ]:
np.random.seed(1234) # for reproducibility, do not modify
y_t = masked_and_blurred_and_noisy_einstein_t

# L2 data fidelity: f(x, y, physics) = ||physics.A(x) - y||_2^2 / 2
data_fidelity = dinv.optim.L2()
# grad f(x, y, A) = A^T(A(x) - y)
print("Data fidelity type:", type(data_fidelity).__name__)
print("f(x, y, A) = ||A(x) - y||_2^2 / 2")

#### Regularization priors

To deal with ill-posed problems, it is common practice to add regularization terms to the optimization problem. In DeepInverse, regularization terms are **priors** of class `deepinv.optim.Prior`. For example, if one wants to define an $\ell_1$ norm penalization to promote sparsity with hyperparameter $\lambda$, one would write:

In [ ]:
lam = 1e2
l1 = dinv.optim.L1Prior()  # g(x) = ||x||_1

print("Prior type:", type(l1).__name__)
print(f"lambda = {lam}")

#### Estimating the Lipschitz constant

Iterative optimization algorithms take steps of size $\tau > 0$ related to the Lipschitz constant of $\nabla f$.
For the L2 data fidelity $f(x) = \|A(x) - y\|_2^2 / 2$, we have $\nabla f(x) = A^\top(A(x) - y)$, whose Lipschitz constant equals $\|A\|^2 = \sigma_{\max}(A)^2$.
We estimate this via power iteration:

In [ ]:
def estimate_lipschitz(physics, input_shape, n_iter=30):
    """Estimate ||A||^2 = sigma_max(A)^2 via power iteration."""
    x = torch.randn(input_shape)
    x /= x.norm()
    with torch.no_grad():
        for _ in range(n_iter):
            Ax = physics.A(x)
            AtAx = physics.A_adjoint(Ax)
            norm = AtAx.norm()
            x = AtAx / norm
    return norm.item()

torch.manual_seed(1234)
lipschitz_A = estimate_lipschitz(physics, (1, 1, 512, 512))
print(f"||A||^2 (Lipschitz constant of grad f): {lipschitz_A:.5f}")

#### Solvers

DeepInverse provides optimization algorithms to solve inverse problems.
Among them is the [**Proximal Gradient Descent (PGD)**](https://deepinv.github.io/deepinv/api/stubs/deepinv.optim.PGD.html), which solves:
$$\hat{x} = \arg\min_x f(x; y, A) + \lambda g(x)$$
where $f$ is the data fidelity and $g$ is the regularization prior with weight $\lambda$.

PGD iterations: $x_{k+1} = \operatorname{prox}_{\gamma \lambda g}\!\left(x_k - \gamma \nabla_x f(x_k; y, A)\right)$

Using solvers is a two-stage process:
1. Define the problem by instantiating a `Solver` model;
2. Solve by calling `model(y, physics, init=x0)`.

In [ ]:
# Define the solver
model = dinv.optim.PGD(
    data_fidelity=data_fidelity,
    prior=l1,
    lambda_reg=lam,
    stepsize=1.0 / lipschitz_A,  # tau = 1 / ||A||^2
    max_iter=50,
    early_stop=False,
)

# Solve with zero initialization
with torch.no_grad():
    x_opt_t = model(y_t, physics, init=torch.zeros_like(y_t))
x_opt = x_opt_t.squeeze().numpy()

In [ ]:
plt.close('all')
view = viewer([x_opt, y_t.squeeze().numpy()], widgets=True, hist=False, axis=True, cmap='gray')

Unsurprisingly, the resulting image is of poor quality as promoting sparsity is not appropriate for this kind of image.

## Hands-on DeepInverse with TV regularization (3 points)

$\ell_1$ regularization is not used much with natural images, contrary to TV regularization which favors piecewise-constant solutions. 

For **2 points**, using DeepInverse and the **Proximal Gradient Descent** algorithm, compute the solution to the following optimization problem **under positivity constraints**:
$$x_\lambda^\ast = \arg \min_{x \geq 0} \| A(x) - y \|_2^2 + \lambda \| \nabla x \|_2^2$$
where $\lambda=0.05$, $A$ is the function defined in the previous section (convolution with PSF of movement followed by partial masking) and $y$ is the blurred, masked and noisy Einstein. Pay attention that, contrary to the previous notebook, the squared $\ell_2$ norm was chosen to ensure differentiability.

**Note:** For reasons of time, stop the algorithm after 50 iterations and take the $y$ vector as initialization (the algorithm should finish in less than 30 seconds).

*Hints:*
    
1. The term $\| A(x) - y \|_2^2 + \lambda \| \nabla x \|_2^2$ is smooth and differentiable. A custom `DataFidelity` subclass `L2WithGradientReg` has been implemented for it. Complete its `grad` method which returns $A^\top(A(x) - y) + \lambda \nabla^\top(\nabla x)$.

2. For the gradient of $\| \nabla x \|_2^2$, you may use the `nabla` and `nabla_adjoint` methods of `self._tv`, which is a `dinv.optim.TVPrior`.

3. The positivity constraint (indicator of the positive orthant) has been implemented as a custom `Prior` subclass `PositiveOrthantPrior`. Complete its `prox` method which clamps values to $[0, +\infty)$.

4. The Lipschitz constant of the combined gradient is bounded by $\|A\|^2 + 8\lambda$ (where 8 is the spectral norm bound for the 2D finite difference operator).

In [ ]:
lam = 5e-2 # Regularization parameter

# Define custom classes
class L2WithGradientReg(dinv.optim.DataFidelity):
    """Data fidelity ||A(x)-y||^2/2 + lam_tv/2 * ||nabla x||^2"""
    def __init__(self, lam_tv=0.0):
        super().__init__()
        self.lam_tv = lam_tv
        self._tv = dinv.optim.TVPrior()
    
    def fn(self, x, y, physics, **kwargs):
        r = physics.A(x) - y
        nabla_x = self._tv.nabla(x)
        return 0.5 * (r**2).sum() + 0.5 * self.lam_tv * (nabla_x**2).sum()
    
    def grad(self, x, y, physics, **kwargs):
        # YOUR CODE HERE

class PositiveOrthantPrior(dinv.optim.Prior):
    """Indicator function of the positive orthant (non-negativity constraint)"""
    def fn(self, x, *args, **kwargs):
        return torch.where(x < 0, torch.tensor(float('inf'), device=x.device), torch.zeros_like(x)).sum()
    
    def prox(self, x, *args, gamma=1.0, **kwargs):
        # YOUR CODE HERE

# Define and fit the solver
# Lipschitz constant upper bound: ||A||^2 + lam * 8
lipschitz_tv = None
# YOUR CODE HERE

data_fidelity_tv = L2WithGradientReg(lam_tv=lam)
pos_prior = PositiveOrthantPrior()

model_tv = dinv.optim.PGD(
    data_fidelity=data_fidelity_tv,
    prior=pos_prior,
    lambda_reg=1.0,  # positivity constraint is scale-free
    stepsize=1.0 / lipschitz_tv,
    max_iter=50,
    early_stop=False,
)
with torch.no_grad():
    x_opt_t = model_tv(y_t, physics, init=y_t.clone())

# Get solution
x_opt = x_opt_t.squeeze().numpy()

plt.close('all')
view = viewer([x_opt, y_t.squeeze().numpy()], widgets=True, hist=False, axis=True, cmap='gray')

**For 1 point**, answer the following MCQ:
* What is the closest to the SNR of the solution found with DeepInverse ? (**1 point**)
1. $20.25$ dB
2. $21.25$ dB
3. $22.25$ dB
4. $23.25$ dB
5. $24.25$ dB
6. $25.25$ dB
7. $26.25$ dB 

*Hint:* You may use the next cell to use or redefine the SNR function that you implemented in the first notebook.

In [ ]:
# YOUR CODE HERE

Modify the variable `answer` in the next cell to reflect your choices.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
if answer not in list(range(1, 8)):
    print('WARNING!\nPossible answers are integers between 1 and 7.')

## Application: tomographic reconstruction (4 points)

Tomographic reconstruction is a computational process used in imaging sciences to reconstruct 2D or 3D images from a series of projections captured at different angles. 

The fundamental principle behind tomographic reconstruction is the Radon transform, mapping a function in 2D or 3D space to a set of line integrals. In the context of medical imaging, these line integrals represent the attenuation of X-rays as they pass through a human body. By capturing these attenuations from various angles, it is possible to reconstruct a cross-sectional image of the internal structure.

The process typically involves the following steps:

- Data Acquisition: Projection data is gathered from different angles, typically by rotating the X-ray source and detector around the object being imaged. The acquired projection data are then 'smeared' back across the imaging field for each angle (back-projection). The data representation that stacks 1D projections of an object at various angles is called a sinogram.

- Reconstruction: Various algorithms, such as Filtered Back-Projection (FBP) or iterative methods, are used to compute the original image from the smeared data. However, due to the non-idealities inherent to the tomographic setup, regularization is often useful to reconstruct the original object.

### Generating ideal sinogram (simulation of real data acquisition)

Given an image (for example `images/MRI.png`), we can simulate the real data acquisition process using the Radon transform.
DeepInverse provides a built-in `Tomography` operator that implements the Radon transform and its inverse (Filtered Back-Projection).
In this example, projection data is gathered from 360 angles between 0 and 180 degrees. The result is an ideal sinogram.

In [ ]:
# Import brain image and downsample for speed
phantom = skimage.io.imread('images/MRI.png').astype(np.float64)[::4, ::4] / 255
# Convert to torch: (1, 1, 128, 128)
phantom_t = torch.tensor(phantom, dtype=torch.float32)[None, None]
nb_angles = 360

In [ ]:
# Radon operator: maps (1,1,128,128) -> (1,1,128,360)
tomography_ideal = dinv.physics.Tomography(
    angles=torch.linspace(0, 180, nb_angles),
    img_width=phantom.shape[0],  # 128
    circle=True,
    normalize=True,
    device='cpu'
)

In [ ]:
# Compute ideal sinogram: shape (1, 1, 128, 360)
sino_ideal_t = tomography_ideal(phantom_t)
sino_ideal = sino_ideal_t.squeeze().numpy()

plt.close('all')
view = viewer([phantom, sino_ideal], widgets=True, hist=False, axis=True, cmap='gray')

### Reconstruction with Filtered Back-Projection (FBP)

In the ideal case, the reconstructed object can be well recovered from its sinogram using the algorithm of Filtered Back-Projection, implemented in DeepInverse via the `fbp` method of the `Tomography` operator.

In [ ]:
fbp_ideal_t = tomography_ideal.fbp(sino_ideal_t)
fbp_ideal = fbp_ideal_t.squeeze().detach().numpy()

plt.close('all')
view = viewer([fbp_ideal, phantom, fbp_ideal - phantom], title=["Ideal FBP", "Phantom", "Difference"], widgets=True, hist=False, axis=True, cmap='gray')

### Reconstruction from a non-ideal sinogram

However, in practice, the process of acquisition deviates from the assumptions usually made by traditional Radon transform models. The complexities of the studied setup in this example include:

- Detector Width: Detectors feature non-negligible widths, hence perform tube integrals in place of line integrals assumed by the Radon transform. This results in a blurring effect in the sinogram.

- Probabilistic Element: There's a chance that detector pixels might malfunction during each scan, introducing missing values in the sinogram.

- X-ray Beam Intensity: The X-ray beam's intensity tapers towards the edges, causing a similar tapering effect in the sinogram, which complicates the application of the standard Radon transform.

Each of the perturbations are described in DeepInverse by the following operators:

In [ ]:
h_sino, w_sino = phantom.shape[0], nb_angles  # 128, 360
img_size_sino = (1, h_sino, w_sino)

# Detector Width: 1D blur along the spatial dimension of the sinogram
class RowBlur(dinv.physics.LinearPhysics):
    """Uniform 3-tap blur along the row dimension (self-adjoint under circular BC)."""
    def A(self, x, **kwargs):
        C = x.shape[1]
        k = torch.ones(C, 1, 3, 1, device=x.device, dtype=x.dtype) / 3
        x_pad = torch.nn.functional.pad(x, (0, 0, 1, 1), mode='circular')
        return torch.nn.functional.conv2d(x_pad, k, groups=C)
    
    def A_adjoint(self, y, **kwargs):
        return self.A(y)  # self-adjoint: symmetric kernel + circular BC

blur = RowBlur()

# Probabilistic Element: random mask on sinogram pixels (1% chance of dropout)
np.random.seed(1234) # for reproducibility, do not modify
mask_sino = np.random.binomial(n=1, p=0.99, size=(h_sino, w_sino))
mask_sino_t = torch.tensor(mask_sino, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
sino_mask = dinv.physics.Inpainting(img_size=img_size_sino, mask=mask_sino_t)

# X-ray Beam Intensity: Hamming taper along the spatial dimension
class HammingTaper(dinv.physics.LinearPhysics):
    """Diagonal operator that applies a Hamming taper along the row dimension."""
    def __init__(self, n_rows, **kwargs):
        super().__init__(**kwargs)
        taper = torch.tensor(np.hamming(n_rows), dtype=torch.float32)
        self.register_buffer('taper', taper.view(1, 1, -1, 1))  # (1, 1, n_rows, 1)
    
    def A(self, x, **kwargs):
        return x * self.taper
    
    def A_adjoint(self, y, **kwargs):
        return y * self.taper  # self-adjoint (real diagonal)

beam_intensity = HammingTaper(n_rows=h_sino)

# Compose all operators: Radon -> blur -> sino_mask -> beam_intensity
tomography_nonideal = dinv.physics.ComposedLinearPhysics(tomography_ideal, blur, sino_mask, beam_intensity)

In [ ]:
sino_nonideal_t = tomography_nonideal(phantom_t)
sino_nonideal = sino_nonideal_t.squeeze().numpy()

plt.close('all')
view = viewer([sino_nonideal, sino_ideal], widgets=True, hist=False, axis=True, cmap='gray')

#### Naive application of Filtered-Back Projection

**For 1 point** answer the following MCQ:
* What is the closest to the SNR of naive application of Filtered-Back Projection to the non-ideal sinogram ? (**1 point**)
1. $1.87$ dB
2. $2.87$ dB
3. $3.87$ dB
4. $4.87$ dB
5. $5.87$ dB
6. $6.87$ dB
7. $7.87$ dB

Use the next cell to compute it.

In [ ]:
fbp_nonideal_t = None
# YOUR CODE HERE

fbp_nonideal = fbp_nonideal_t.squeeze().detach().numpy()
plt.close('all')
view = viewer([fbp_nonideal, phantom, fbp_nonideal - phantom], title=["Non-ideal FBP", "Phantom", "Difference"], widgets=True, hist=False, axis=True, cmap='gray')

Modify the variable `answer` in the next cell to reflect your choices. 

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
if answer not in list(range(1, 8)):
    print('WARNING!\nPossible answers are integers between 1 and 7.')

#### Leveraging DeepInverse for regularization

For **2 points**, solve the reconstruction problem **under positive constraints**:
$$x^\ast = \arg \min_{x \geq 0} \| A(x) - y \|_2^2 + \lambda  \| x \|_1 + \mu \| \nabla x \|_2^2$$
with $\lambda=1.3 \times 10^{-3}$ and $\mu=10^{-4}$, and where $y$ represents the non-ideal sinogram and $A$ is the composition of the Radon transform followed by all the three perturbations defined previously.

Use the **Proximal Gradient Descent** solver from DeepInverse with the normalized tomography operator and tune the solver parameters so that the reconstruction matches the Pyxu scale while keeping the step size fixed.

**Note:** For reasons of time, stop the algorithm after 200 iterations and take the all-zeros vector as initialization (the algorithm should finish in less than 2 minutes).

*Hints:*
    
1. The `dinv.optim.Prior` subclass `PositiveL1Prior` implements the L1 prior with positive constraints. Complete its `prox` method which implements positive soft thresholding: $\operatorname{prox}_{\gamma g}(x) = \max(x - \gamma, 0)$.

2. Combine the L2 data fidelity and the gradient regularization ($\mu \|\nabla x\|_2^2$) into a single `L2WithGradientReg` (already defined in the previous section).

3. The gradient step size is defined with the argument `stepsize` in the `PGD` constructor.

In [ ]:
y_tomo_t = sino_nonideal_t
lam = 1.3e-3 # regularization parameter for L1
mu = 1e-4 # regularization parameter for gradient L2

# Define the operators
class PositiveL1Prior(dinv.optim.Prior):
    """L1 norm with positivity constraint: prox = positive soft thresholding."""
    def fn(self, x, *args, **kwargs):
        return x.clamp(min=0).sum()
    
    def prox(self, x, *args, gamma=1.0, **kwargs):
        # YOUR CODE HERE

data_fidelity_tomo = L2WithGradientReg(lam_tv=mu)
prior_tomo = PositiveL1Prior()

# Define and fit the solver with all-zeros vector as initialization
model_tomo = dinv.optim.PGD(
    data_fidelity=data_fidelity_tomo,
    prior=prior_tomo,
    lambda_reg=lam,
    stepsize=2.0,
    max_iter=200,
    early_stop=False,
)
with torch.no_grad():
    x_opt_t = model_tomo(y_tomo_t, tomography_nonideal, init=torch.zeros_like(fbp_nonideal_t))

# Get solution
x_opt_tomo = x_opt_t.squeeze().numpy()

# Display the solution
plt.close('all')
view = viewer([x_opt_tomo, fbp_nonideal, fbp_ideal, phantom], title=["DeepInv solution", "Non-ideal FBP", "Ideal FBP", "Phantom"], widgets=True, hist=False, axis=True, cmap='gray')

For **1 point** answer the following MCQ:
* What is the closest to the SNR of the reconstructed image ? (**1 point**)
1. $1.46$ dB
2. $2.46$ dB
3. $3.46$ dB
4. $4.46$ dB
5. $5.46$ dB
6. $6.46$ dB
7. $7.46$ dB

Use the next cell to compute it. 

In [ ]:
# YOUR CODE HERE

Modify the variable `answer` in the next cell to reflect your choices.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
if answer not in list(range(1, 8)):
    print('WARNING!\nPossible answers are integers between 1 and 7.')

🎉 Congratulations on finishing the second part of the Inverse Problem lab!

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *2_deepinv.ipynb*,
* Name the zip file: *inverse_problem_lab.zip*.